# FFB Pipeline v2 — Temporal Accumulation + Semantic Segmentation

Automated mass estimation of oil palm Fresh Fruit Bunches (FFBs) from a single
Intel RealSense D455 depth camera. Zero-shot; no fine-tuning required.

| Approach | Segmentation | Depth source | In-sample MAE | r² |
|---|---|---|---|---|
| **v1** (baseline) | Depth foreground mask, 15 frames | Per-frame median | 1.80 kg | 0.686 |
| **A** (best) | Depth mask + adaptive HSV/z_window | Temporal nanmedian, 120 frames | **1.20 kg** | **0.863** |
| **C** | Grounding DINO → SAM2 | Temporal nanmedian, 120 frames | 1.89 kg | 0.747 |

4/7 exhaustive CV (Approach A, excl. FFB18): MAE=1.67 kg · r²=0.893 (per-FFB mean)  
Aqil thesis benchmark (n=50, manual segmentation): r²=0.900

Run cells top-to-bottom. `run_all` reads from `fused_cache/` on subsequent runs.

In [ ]:
# ── 0. Pre-flight ─────────────────────────────────────────────────────────
import os, sys, pathlib

PROJECT_DIR = '/kaggle/input/datasets/rajulkabir/rgbd-mass-ffb'

for candidate in [
    f'{PROJECT_DIR}/data',
    f'{PROJECT_DIR}/data/data',
    PROJECT_DIR,
]:
    subdirs = [d for d in os.listdir(candidate)
               if os.path.isdir(os.path.join(candidate, d)) and d.startswith('FFB')]
    if subdirs:
        DATA_DIR = candidate
        break
else:
    raise RuntimeError('Cannot find FFB folders under ' + PROJECT_DIR)

sys.path.insert(0, PROJECT_DIR)

for f in ('perception_pipeline.py', 'bag_reader.py', 'ground_truth.csv', 'validation_density.csv'):
    status = 'OK   ' if (pathlib.Path(PROJECT_DIR) / f).exists() else 'MISSING'
    print(f'  {status}  {f}')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

bundles = sorted(d for d in os.listdir(DATA_DIR)
                 if os.path.isdir(os.path.join(DATA_DIR, d)) and d.startswith('FFB'))
print(f'DATA_DIR : {DATA_DIR}')
print(f'Bundles  : {bundles}')
print(f'Device   : {DEVICE}  ({N_GPUS} GPU(s))')
if torch.cuda.is_available():
    for i in range(N_GPUS):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB')

In [ ]:
# ── 1. Install dependencies ────────────────────────────────────────────────
import subprocess

def pip(*args, check=True):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args],
                       capture_output=True, text=True)
    if check and r.returncode != 0:
        print(r.stdout[-2000:]); print(r.stderr[-2000:])
        raise RuntimeError(f'pip failed: {args}')
    return r.returncode == 0

pip('ultralytics>=8.2');  print('ultralytics OK')

sam2_ok = (
    pip('sam2', check=False) or
    pip('git+https://github.com/facebookresearch/segment-anything-2.git@v1.0', check=False) or
    pip('git+https://github.com/facebookresearch/segment-anything-2.git@main', check=False) or
    pip('git+https://github.com/facebookresearch/segment-anything-2.git', check=False)
)
if not sam2_ok: raise RuntimeError('SAM2 install failed')
print('SAM2 OK')

pip('pyrealsense2');               print('pyrealsense2 OK')
pip('transformers>=4.38', check=False) and print('transformers OK')

# Transformers and related packages may downgrade numpy from 2.x to 1.x,
# which breaks the Kaggle-preinstalled scipy (compiled for numpy 2.x).
pip('numpy>=2.0', check=False)
print('numpy pinned >=2.0 (scipy ABI fix)')

print('\nAll dependencies installed.')

In [ ]:
# ── 2. Download model weights ──────────────────────────────────────────────
import urllib.request, shutil, pathlib

_clip_cache = pathlib.Path.home() / '.cache' / 'clip'
if _clip_cache.exists():
    shutil.rmtree(_clip_cache)
    print(f'  cleared CLIP cache: {_clip_cache}')

WEIGHTS = {
    'sam2_hiera_small.pt': 'https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt',
    'yolov8m-world.pt':    'https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8m-world.pt',
}
for fname, url in WEIGHTS.items():
    if os.path.exists(fname):
        print(f'  cached: {fname}')
    else:
        print(f'  downloading {fname} ...')
        urllib.request.urlretrieve(url, fname)
        print(f'  done ({os.path.getsize(fname)/1e6:.1f} MB)')

In [ ]:
# ── 3. Pipeline patches (identical to v1, except _auto_margin gains clip params) ──
from perception_pipeline import (
    FFBPerceptionPipeline, CameraIntrinsics, PerceptionResult,
    BundleResult, process_bundle_multi_frame,
)
import perception_pipeline as _pp
from bag_reader import find_bundle_bags, get_depth_scale, get_intrinsics_from_bag
import numpy as np, cv2, glob as _glob
import matplotlib.pyplot as plt
import pandas as pd


def _patched_load_sam2(config, checkpoint, device='cpu'):
    import torch._jit_internal as _jit_int
    if not getattr(_jit_int, '_clear_fn_overloads_patched', False):
        _orig_clear = _jit_int._clear_fn_overloads
        def _safe_clear_fn_overloads(qual_name):
            try: _orig_clear(qual_name)
            except KeyError: pass
        _jit_int._clear_fn_overloads = _safe_clear_fn_overloads
        _jit_int._clear_fn_overloads_patched = True
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    bn  = os.path.basename(config)
    bne = os.path.splitext(bn)[0]
    candidates = dict.fromkeys([
        config, bn, bne,
        bn.replace('sam2_', 'sam2.1_'), bne.replace('sam2_', 'sam2.1_'),
        bn.replace('sam2.1_', 'sam2_'), bne.replace('sam2.1_', 'sam2_'),
    ])
    import sam2 as _sam2mod
    disk = _glob.glob(os.path.join(os.path.dirname(_sam2mod.__file__), '**', '*.yaml'), recursive=True)
    for d in disk:
        candidates[d] = candidates[os.path.basename(d)] = candidates[os.path.splitext(os.path.basename(d))[0]] = None
    for cfg in candidates:
        try:
            model = build_sam2(cfg, checkpoint, device=device)
            print(f'  SAM2 loaded with config: {cfg}'); break
        except Exception: pass
    else:
        raise RuntimeError(f'SAM2: no working config found. Disk configs: {disk}')
    pred = SAM2ImagePredictor(model)
    # SAM2 defaults to bfloat16/autocast on GPU; always force fp32 so that
    # mixed-dtype matmul errors don't occur when called from Approach C.
    pred.model = pred.model.float()
    return pred

_pp._load_sam2 = _patched_load_sam2


def _patched_detect(self, rgb: np.ndarray):
    import torch as _torch
    h, w = rgb.shape[:2]
    with _torch.inference_mode():
        results = self.yolo.predict(rgb, verbose=False, conf=0.01)
    boxes = results[0].boxes
    if boxes is None or len(boxes) == 0:
        margin_x, margin_y = w * 0.25, h * 0.25
        return [margin_x, margin_y, w - margin_x, h - margin_y], 0.0
    confidences = boxes.conf.cpu().numpy()
    best = int(np.argmax(confidences))
    return boxes.xyxy[best].cpu().numpy().tolist(), float(confidences[best])

FFBPerceptionPipeline._detect = _patched_detect


def _auto_margin(depth_m, z_front, valid,
                 search_lo=0.05, search_hi=0.45, n_bins=80, fallback_m=0.10,
                 clip_lo=0.08, clip_hi=0.22):
    """
    Detect tarp-to-FFB protrusion height from depth histogram.
    clip_lo / clip_hi parameterised so callers can pass wide bounds for
    multi-frame margin sampling and narrow bounds for final application.
    """
    band = depth_m[valid & (depth_m > z_front + search_lo)
                        & (depth_m < z_front + search_hi)]
    if band.size < 200:
        return fallback_m
    hist, edges = np.histogram(band, bins=n_bins)
    z_tarp = float(edges[int(np.argmax(hist))])
    return float(np.clip(z_tarp - z_front - 0.05, clip_lo, clip_hi))


def _depth_foreground_mask(depth_m, rgb=None,
                           min_area_frac=0.003, max_area_frac=0.35,
                           colour_s_min=40, colour_v_max=160):
    H, W  = depth_m.shape
    valid = (depth_m > 0.1) & (depth_m < 10.0)
    if not valid.any():
        return None, None, None
    z_front  = float(np.percentile(depth_m[valid], 5))
    margin_m = _auto_margin(depth_m, z_front, valid)
    fg = (valid & (depth_m <= z_front + margin_m)).astype(np.uint8)
    if rgb is not None:
        rgb_d = cv2.resize(rgb, (W, H), interpolation=cv2.INTER_LINEAR) if rgb.shape[:2] != (H, W) else rgb
        hsv   = cv2.cvtColor(rgb_d, cv2.COLOR_RGB2HSV)
        fg    = (fg.astype(bool)
                 & (hsv[..., 1] >= colour_s_min)
                 & (hsv[..., 2] <= colour_v_max)).astype(np.uint8)
    k  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    fg = cv2.morphologyEx(cv2.morphologyEx(fg, cv2.MORPH_CLOSE, k), cv2.MORPH_OPEN, k)
    contours, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, None, None
    img_area       = H * W
    cx_img, cy_img = W * 0.5, H * 0.5
    half_diag      = (cx_img**2 + cy_img**2) ** 0.5
    def _dist_centre(cnt):
        M = cv2.moments(cnt)
        if M['m00'] == 0: return float('inf')
        return ((M['m10']/M['m00'] - cx_img)**2 + (M['m01']/M['m00'] - cy_img)**2) ** 0.5
    sized = [c for c in contours
             if min_area_frac * img_area < cv2.contourArea(c) < max_area_frac * img_area]
    if not sized:
        return None, None, None
    c = min(sized, key=_dist_centre)
    if _dist_centre(c) > 0.75 * half_diag:
        c = max(sized, key=cv2.contourArea)
        if _dist_centre(c) > 0.75 * half_diag:
            return None, None, None
    mask = np.zeros((H, W), dtype=np.uint8)
    cv2.drawContours(mask, [c], -1, 1, thickness=cv2.FILLED)
    if mask.mean() > max_area_frac:
        return None, None, None
    x, y, w, h = cv2.boundingRect(c)
    return mask.astype(bool), [float(x), float(y), float(x+w), float(y+h)], margin_m


def _expand_mask_bbox(fg_mask, depth_m, rgb_image, z_front,
                      z_window=0.25, s_min=25, v_max=160, pad_px=20):
    dh, dw = depth_m.shape
    if not fg_mask.any():
        return fg_mask
    ys, xs = np.where(fg_mask)
    y1 = max(0,  ys.min() - pad_px);  y2 = min(dh, ys.max() + pad_px)
    x1 = max(0,  xs.min() - pad_px);  x2 = min(dw, xs.max() + pad_px)
    search = np.zeros((dh, dw), dtype=bool)
    search[y1:y2, x1:x2] = True
    depth_ok = (depth_m > 0) & (depth_m <= z_front + z_window)
    if rgb_image is not None:
        rh, rw = rgb_image.shape[:2]
        rgb_d  = cv2.resize(rgb_image, (dw, dh), interpolation=cv2.INTER_LINEAR) if (rh, rw) != (dh, dw) else rgb_image
        hsv       = cv2.cvtColor(rgb_d, cv2.COLOR_RGB2HSV)
        colour_ok = (hsv[..., 1] >= s_min) & (hsv[..., 2] <= v_max)
    else:
        colour_ok = np.ones((dh, dw), dtype=bool)
    expanded = search & depth_ok & colour_ok
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    expanded = cv2.morphologyEx(expanded.astype(np.uint8), cv2.MORPH_OPEN, k).astype(bool)
    return expanded if expanded.any() else fg_mask


SCALE_2D5 = 2.02
SCALE_BY_WIDTH = {1280: 2.02, 848: 2.53}


def _compute_2d5_volume(point_cloud: np.ndarray, grid_step: float = 0.002) -> tuple:
    if point_cloud.shape[0] < 20:
        return 0.0, 0.0
    x, y, z = point_cloud.T.astype(np.float64)
    z_ref    = float(np.percentile(z, 95))
    x0, y0  = float(x.min()), float(y.min())
    xi = np.floor((x - x0) / grid_step).astype(np.int32)
    yi = np.floor((y - y0) / grid_step).astype(np.int32)
    ny = int(yi.max()) + 1
    z_top = np.full((int(xi.max())+1) * ny, z_ref, dtype=np.float64)
    np.minimum.at(z_top, xi * ny + yi, z)
    has_data = z_top < (z_ref - 1e-4)
    return float(np.sum(z_ref - z_top[has_data]) * grid_step**2), z_ref


FFBPerceptionPipeline._compute_convex_volume = staticmethod(
    lambda pc: _compute_2d5_volume(pc)[0] * SCALE_2D5)


def _patched_process_frame(self, rgb_image, depth_map, camera_intrinsics):
    depth_m = FFBPerceptionPipeline._to_metric_depth(depth_map, camera_intrinsics.depth_scale)
    depth_m = cv2.medianBlur(depth_m, 3)
    dh, dw  = depth_m.shape
    rh, rw  = rgb_image.shape[:2]
    cam_width = int(getattr(camera_intrinsics, 'width', 0))
    scale     = SCALE_BY_WIDTH.get(cam_width, SCALE_2D5)
    fg_mask, _, _ = _depth_foreground_mask(depth_m, rgb=rgb_image)
    if fg_mask is not None:
        z_front = float(np.percentile(depth_m[depth_m > 0.1], 5))
        if cam_width == 848:
            mask_depth_res = _expand_mask_bbox(fg_mask, depth_m, rgb_image, z_front)
            expand_tag = 'expanded'
        else:
            mask_depth_res = fg_mask
            expand_tag = 'tight'
        confidence = 0.95
        print(f'    [depth]  z_front={z_front:.3f}m  mask={mask_depth_res.mean()*100:.1f}% ({expand_tag})  scale={scale}')
    else:
        bbox_yolo, confidence = FFBPerceptionPipeline._detect(self, rgb_image)
        x1, y1, x2, y2 = bbox_yolo
        if (x2-x1)*(y2-y1) / (rh*rw) > 0.40:
            print('    [yolo-skip]  box too large')
            return PerceptionResult(
                mask=np.zeros((dh, dw), dtype=bool),
                point_cloud=np.zeros((0, 3), dtype=np.float32),
                estimated_convex_volume=0.0, detection_confidence=0.0, depth_fill_ratio=0.0)
        print(f'    [yolo-sam2]  bbox={[int(v) for v in bbox_yolo]}  conf={confidence:.2f}')
        mask_rgb = self._segment(rgb_image, bbox_yolo)
        mask_depth_res = (cv2.resize(mask_rgb.astype(np.uint8), (dw, dh),
                                     interpolation=cv2.INTER_NEAREST).astype(bool)
                          if (rh, rw) != (dh, dw) else mask_rgb)
        z_front = float(np.percentile(depth_m[depth_m > 0.1], 5))
    fill_ratio = 0.0
    if self.use_depth_fusion and self.depth_model is not None:
        rgb_fuse = cv2.resize(rgb_image, (dw, dh)) if (rh, rw) != (dh, dw) else rgb_image
        depth_m, fill_ratio = self._fuse_depth(rgb_fuse, depth_m, mask_depth_res)
    point_cloud      = self._project_to_3d(depth_m, mask_depth_res, camera_intrinsics)
    v_2d5_raw, z_ref = _compute_2d5_volume(point_cloud)
    volume           = v_2d5_raw * scale if v_2d5_raw > 0 else 0.0
    print(f'    [vol]  2d5_raw={v_2d5_raw*1000:.2f}L  x{scale:.2f}={volume*1000:.2f}L  dome={(z_ref-z_front)*100:.1f}cm')
    return PerceptionResult(
        mask=mask_depth_res, point_cloud=point_cloud,
        estimated_convex_volume=volume, detection_confidence=float(confidence),
        depth_fill_ratio=float(fill_ratio))

FFBPerceptionPipeline.process_frame = _patched_process_frame


@staticmethod
def _patched_project_to_3d(depth_m, mask, K):
    h, w   = depth_m.shape
    us, vs = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32))
    valid  = mask & (depth_m > 0)
    Z = depth_m[valid].astype(np.float32)
    u = us[valid]; v = vs[valid]
    if Z.size > 0:
        keep    = Z <= float(np.percentile(Z, 5)) + 0.25
        Z, u, v = Z[keep], u[keep], v[keep]
    X = (u - K.cx) * Z / K.fx
    Y = (v - K.cy) * Z / K.fy
    return np.stack([X, Y, Z], axis=-1)

FFBPerceptionPipeline._project_to_3d = _patched_project_to_3d


GT = pd.read_csv(f'{PROJECT_DIR}/ground_truth.csv')
GT.index = GT['FFB_No'].astype(int)

def gt(ffb_name):
    num = int(ffb_name.replace('FFB', ''))
    return GT.loc[num].to_dict() if num in GT.index else {}

DENSITY_CONSTANT = GT['True_Density_kg_L'].mean() * 1000
print(f'Density constant: {DENSITY_CONSTANT:.2f} kg/m3')

val = pd.read_csv(f'{PROJECT_DIR}/validation_density.csv')
val['pred_mass_kg'] = DENSITY_CONSTANT * val['Estimated_Volume_m3']
val['abs_err']      = (val['pred_mass_kg'] - val['Target_Mass_kg']).abs()
val['pct_err']      = 100 * val['abs_err'] / val['Target_Mass_kg']
_val_mae  = val['abs_err'].mean()
_val_mape = val['pct_err'].mean()
print(f'Density validation (n=10):  MAE={_val_mae:.2f} kg  MAPE={_val_mape:.1f}%')
print('All pipeline patches applied.')

In [ ]:
# ── 3c. process_bundle_multi_frame patch (identical to v1) ─────────────────
import perception_pipeline as _pp_mod

def _patched_process_bundle(ffb_dir, pipeline, camera_intrinsics,
                            n_frames=5, max_scan=60):
    from bag_reader import iter_bundle
    candidates = []
    for rgb, depth in iter_bundle(ffb_dir, max_frames=max_scan):
        h, w  = depth.shape
        crop  = depth[h//4: 3*h//4, w//4: 3*w//4]
        candidates.append((int((crop > 0).sum()), rgb.copy(), depth.copy()))
    candidates.sort(key=lambda x: x[0], reverse=True)
    frame_data = []
    for score, rgb, depth in candidates[:min(n_frames, len(candidates))]:
        result = pipeline.process_frame(rgb, depth, camera_intrinsics)
        frame_data.append((score, result.estimated_convex_volume, result))
    if not frame_data:
        return _pp_mod.BundleResult(
            median_volume=0.0, std_volume=0.0, mean_volume=0.0,
            merged_volume=0.0, per_frame_volumes=[],
            best_point_cloud=np.zeros((0, 3)),
            best_mask=np.zeros((0, 0), dtype=bool), n_frames_used=0)
    vols = np.array([v for _, v, _ in frame_data], dtype=np.float64)
    keep_mask = np.ones(len(vols), dtype=bool)
    if len(vols) >= 4:
        q1, q3 = np.percentile(vols, [25, 75])
        iqr = q3 - q1
        if iqr > 0:
            keep_mask = (vols >= q1 - 1.5*iqr) & (vols <= q3 + 1.5*iqr)
            if not keep_mask.any(): keep_mask = np.ones(len(vols), dtype=bool)
    kept = [frame_data[i] for i in range(len(frame_data)) if keep_mask[i]]
    dropped = len(frame_data) - len(kept)
    if dropped:
        drop_str = [round(frame_data[i][1]*1000, 1) for i in range(len(frame_data)) if not keep_mask[i]]
        print(f'    [iqr]  dropped {dropped}: {drop_str}L')
    kept_vols = np.array([v for _, v, _ in kept], dtype=np.float64)
    _, _, best_result = max(kept, key=lambda x: x[0])
    all_pts = [r.point_cloud for _, _, r in kept if r.point_cloud.shape[0] > 0]
    merged_vol = 0.0
    if all_pts:
        merged_pts = np.concatenate(all_pts, axis=0)
        v_raw, _   = _compute_2d5_volume(merged_pts)
        merged_vol = v_raw * SCALE_2D5 if v_raw > 0 else 0.0
    return _pp_mod.BundleResult(
        median_volume=float(np.median(kept_vols)), std_volume=float(np.std(kept_vols)),
        mean_volume=float(np.mean(kept_vols)), merged_volume=merged_vol,
        per_frame_volumes=[v for _, v, _ in frame_data],
        best_point_cloud=best_result.point_cloud, best_mask=best_result.mask,
        n_frames_used=len(frame_data))

_pp_mod.process_bundle_multi_frame = _patched_process_bundle
print('process_bundle_multi_frame patched.')

In [ ]:
# ── 4. Pipeline factory ────────────────────────────────────────────────────
import threading
import torch.nn as _nn

_init_lock = threading.Lock()

if not getattr(_nn.Embedding.forward, '_device_safe_patched', False):
    _orig_emb_fwd = _nn.Embedding.forward
    def _device_safe_emb_fwd(self, input):
        if input.device != self.weight.device:
            input = input.to(self.weight.device)
        return _orig_emb_fwd(self, input)
    _device_safe_emb_fwd._device_safe_patched = True
    _nn.Embedding.forward = _device_safe_emb_fwd

PIPELINE_KWARGS = dict(
    yolo_weights='yolov8m-world.pt',
    sam2_config='sam2_hiera_small.yaml',
    sam2_checkpoint='sam2_hiera_small.pt',
    use_depth_fusion=False,
)
YOLO_CLASSES = ['fruit bunch']


def make_pipeline(device):
    import torch as _t
    with _init_lock:
        p = FFBPerceptionPipeline(**PIPELINE_KWARGS, device=device)
    p.yolo.set_classes(YOLO_CLASSES)
    for _attr in ('txt_feats', 'text_feats'):
        for _obj in (p.yolo.model, getattr(p.yolo.model, 'model', None)):
            if _obj is None: continue
            _feat = getattr(_obj, _attr, None)
            if isinstance(_feat, _t.Tensor) and _feat.device.type == 'cpu':
                try: setattr(_obj, _attr, _feat.to(device))
                except Exception: pass
    print(f'  Pipeline ready on {device}')
    return p

In [ ]:
# ── 4b. Grounding DINO detector (replaces YOLO-World in detection fallback) ─
# Kept in fp32; torch.autocast handles mixed precision for the forward pass.
# This avoids the fp32/fp16 dtype mismatch that occurs when BERT's text encoder
# silently upcasts intermediate activations even after a .half() model cast.

_gdino_model = _gdino_processor = None
GDINO_AVAILABLE = False
_GDINO_BOX_THRESH_KEY = 'box_threshold'


def _try_load_gdino():
    global _gdino_model, _gdino_processor, GDINO_AVAILABLE, _GDINO_BOX_THRESH_KEY
    try:
        import inspect
        from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
        _id = 'IDEA-Research/grounding-dino-base'
        _gdino_processor = AutoProcessor.from_pretrained(_id)
        # Keep in fp32 — autocast in _gdino_detect handles fp16 ops selectively
        _gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(_id).to(DEVICE)
        _gdino_model.eval()
        _sig = inspect.signature(
            _gdino_processor.post_process_grounded_object_detection)
        _GDINO_BOX_THRESH_KEY = ('box_threshold'
                                  if 'box_threshold' in _sig.parameters
                                  else 'threshold')
        GDINO_AVAILABLE = True
        print(f'  Grounding DINO loaded on {DEVICE} (fp32 + autocast)  '
              f'(box param: "{_GDINO_BOX_THRESH_KEY}")')
    except Exception as e:
        print(f'  Grounding DINO unavailable: {e}')
        print('  Detection fallback: YOLO-World only.')

_try_load_gdino()

_GDINO_PROMPTS = [
    'fresh oil palm fruit bunch.',
    'palm fruit bunch.',
    'fruit bunch.',
]
_GDINO_BOX_THRESH = 0.15
_GDINO_EARLY_STOP = 0.50


def _gdino_detect(rgb: np.ndarray, verbose=False):
    """Returns (bbox_xyxy, conf) or None."""
    from PIL import Image as _PIL
    import torch as _t
    h, w  = rgb.shape[:2]
    pil   = _PIL.fromarray(rgb)
    best_box, best_conf = None, 0.0

    # autocast lets PyTorch decide per-op whether fp16 is safe,
    # avoiding the manual-cast dtype mismatches that hit BERT text encoder.
    _ac = (_t.autocast(device_type='cuda', dtype=_t.float16)
           if DEVICE != 'cpu'
           else _t.autocast(device_type='cpu', enabled=False))

    for prompt in _GDINO_PROMPTS:
        inp = _gdino_processor(images=pil, text=prompt,
                               return_tensors='pt').to(DEVICE)
        with _t.no_grad(), _ac:
            out = _gdino_model(**inp)
        res = _gdino_processor.post_process_grounded_object_detection(
            out, inp.input_ids,
            **{_GDINO_BOX_THRESH_KEY: 0.01},
            text_threshold=0.01,
            target_sizes=[(h, w)]
        )[0]
        scores = res['scores'].cpu().numpy()
        if not len(scores):
            if verbose: print(f'      [gdino] "{prompt}"  — no boxes returned')
            continue
        top = float(scores.max())
        if verbose: print(f'      [gdino] "{prompt}"  top={top:.3f}  n={len(scores)}')
        if top > best_conf:
            best_conf = top
            best_box  = res['boxes'][int(np.argmax(scores))].cpu().numpy().tolist()
        if best_conf >= _GDINO_EARLY_STOP:
            break

    if best_box is not None and best_conf >= _GDINO_BOX_THRESH:
        return best_box, best_conf
    if verbose:
        print(f'      [gdino] best conf={best_conf:.3f} < {_GDINO_BOX_THRESH} — no detection')
    return None


def _patched_detect_gdino(self, rgb: np.ndarray):
    """Grounding DINO primary → YOLO-World fallback."""
    import torch as _t
    h, w    = rgb.shape[:2]
    default = [w * 0.25, h * 0.25, w * 0.75, h * 0.75]

    if GDINO_AVAILABLE:
        result = _gdino_detect(rgb)
        if result is not None:
            box, conf = result
            print(f'    [gdino]  bbox={[int(v) for v in box]}  conf={conf:.2f}')
            return box, conf
        print('    [gdino]  no detection — trying YOLO-World')

    with _t.inference_mode():
        yolo_res = self.yolo.predict(rgb, verbose=False, conf=0.01)
    boxes = yolo_res[0].boxes
    if boxes is None or len(boxes) == 0:
        return default, 0.0
    confs = boxes.conf.cpu().numpy()
    best  = int(np.argmax(confs))
    return boxes.xyxy[best].cpu().numpy().tolist(), float(confs[best])


FFBPerceptionPipeline._detect = _patched_detect_gdino


# ── SAM2 fp32 guard ───────────────────────────────────────────────────────────
# SAM2 checkpoint on Kaggle is fp32; no cast needed. Guard defensively.
import torch as _t
_orig_segment = FFBPerceptionPipeline._segment

def _fp32_segment(self, rgb, bbox):
    _bad = [n for n, p in self.sam2.model.named_parameters()
            if p.dtype == _t.float16]
    if _bad:
        self.sam2.model = self.sam2.model.float()
    return _orig_segment(self, rgb, bbox)

FFBPerceptionPipeline._segment = _fp32_segment

print(f'Detection: {"Grounding DINO (fp32+autocast) → YOLO-World" if GDINO_AVAILABLE else "YOLO-World only"}')
print(f'GDINO_AVAILABLE = {GDINO_AVAILABLE}')
print('SAM2 _segment guarded against fp16 params')

## Approaches

Cells below define helper functions for Approaches A and C. Nothing runs until `run_all`.

In [ ]:
# ── Approach A: Temporal depth accumulation ───────────────────────────────
from bag_reader import iter_bundle
import pathlib as _pl


FUSED_CACHE_DIR = 'fused_cache'


def accumulate_bundle_depth(ffb_dir, K, max_frames=120, cache_dir=FUSED_CACHE_DIR):
    """
    Return (fused_depth_m, best_rgb). fused_depth is nanmedian over all frames.

    Results are cached to {cache_dir}/{ffb_name}_fused.npz so re-runs skip
    bag reading entirely. Delete the .npz files (or set CLEAR_CACHE=True in
    run_all) to force recomputation (e.g. after changing max_frames).
    """
    ffb_name   = _pl.Path(ffb_dir).name
    cache_path = _pl.Path(cache_dir) / f'{ffb_name}_fused.npz'

    if cache_path.exists():
        data     = np.load(str(cache_path))
        fused    = data['fused']
        best_rgb = data['rgb']
        print(f'    [accum]  loaded from cache  ({fused.shape[1]}×{fused.shape[0]},'
              f'  valid={100*(fused>0).mean():.1f}%)')
        return fused, best_rgb

    depth_frames = []
    candidates   = []
    for rgb, depth in iter_bundle(ffb_dir, max_frames=max_frames):
        dm = FFBPerceptionPipeline._to_metric_depth(depth, K.depth_scale)
        dm = cv2.medianBlur(dm, 3)
        h, w  = dm.shape
        score = int((dm[h//4:3*h//4, w//4:3*w//4] > 0).sum())
        candidates.append((score, rgb.copy()))
        arr = dm.astype(np.float32)
        arr[arr == 0] = np.nan
        depth_frames.append(arr)
    if not depth_frames:
        return None, None

    stack = np.stack(depth_frames, axis=0)
    with np.errstate(all='ignore'):   # suppress all-NaN slice warning (pixels
        fused = np.nanmedian(stack, axis=0).astype(np.float32)  # with 0 valid frames → nan → 0)
    fused    = np.nan_to_num(fused, nan=0.0)
    best_rgb = max(candidates, key=lambda x: x[0])[1]

    single_pct = 100.0 * float(np.isfinite(depth_frames[0]).mean())
    fused_pct  = 100.0 * float((fused > 0).mean())
    print(f'    [accum]  frames={len(depth_frames)}  '
          f'valid px: {single_pct:.1f}% -> {fused_pct:.1f}%')

    _pl.Path(cache_dir).mkdir(parents=True, exist_ok=True)
    np.savez_compressed(str(cache_path), fused=fused, rgb=best_rgb)
    print(f'    [accum]  cached → {cache_path}')

    return fused, best_rgb


def _border_hsv_stats(rgb, border_px=25):
    h, w = rgb.shape[:2]
    hsv  = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV).astype(float)
    border = np.concatenate([
        hsv[:border_px, :].reshape(-1, 3),
        hsv[h-border_px:, :].reshape(-1, 3),
        hsv[:, :border_px].reshape(-1, 3),
        hsv[:, w-border_px:].reshape(-1, 3),
    ], axis=0)
    s_min = float(np.clip(np.percentile(border[:, 1], 85), 15, 70))
    v_max = float(np.clip(np.percentile(border[:, 2], 95) + 40, 120, 220))
    return s_min, v_max


def _extract_mask(fused_depth, best_rgb, pipeline, K):
    """Returns (mask, z_front, z_window). Shared by approaches A and C."""
    cam_width = int(getattr(K, 'width', 0))

    s_min, v_max = _border_hsv_stats(best_rgb)
    print(f'    [adaptive HSV]  s_min={s_min:.0f}  v_max={v_max:.0f}')

    fg_mask, _, margin_m = _depth_foreground_mask(
        fused_depth, rgb=best_rgb, colour_s_min=s_min, colour_v_max=v_max)

    z_front = float(np.percentile(fused_depth[fused_depth > 0.1], 5))

    if fg_mask is not None:
        z_window = float(margin_m) + 0.05
        print(f'    [adaptive z_window]  margin={margin_m*100:.1f}cm  '
              f'z_window={z_window*100:.1f}cm')
        if cam_width == 848:
            ys, xs    = np.where(fg_mask)
            bbox_diag = float(((xs.max()-xs.min())**2 + (ys.max()-ys.min())**2)**0.5)
            pad       = int(np.clip(0.08 * bbox_diag, 8, 40))
            print(f'    [adaptive pad]  bbox_diag={bbox_diag:.0f}px  pad={pad}px')
            mask = _expand_mask_bbox(fg_mask, fused_depth, best_rgb, z_front,
                                     z_window=z_window, pad_px=pad)
        else:
            mask = fg_mask
        tag = 'expanded' if cam_width == 848 else 'tight'
        print(f'    [depth]  z_front={z_front:.3f}m  mask={mask.mean()*100:.1f}% ({tag})')
    else:
        detector_tag = 'gdino-sam2' if GDINO_AVAILABLE else 'yolo-sam2'
        bbox, _ = FFBPerceptionPipeline._detect(pipeline, best_rgb)
        mask_rgb = pipeline._segment(best_rgb, bbox)
        dh, dw   = fused_depth.shape
        rh, rw   = best_rgb.shape[:2]
        mask = (cv2.resize(mask_rgb.astype(np.uint8), (dw, dh),
                           interpolation=cv2.INTER_NEAREST).astype(bool)
                if (rh, rw) != (dh, dw) else mask_rgb)
        z_window = 0.25
        print(f'    [{detector_tag} fallback]  z_window=25cm (default)')

    return mask, z_front, z_window


def process_bundle_accumulated(ffb_dir, pipeline, K,
                                fused_depth=None, best_rgb=None):
    """Approach A: single-pass on temporally accumulated depth.
    Pass fused_depth/best_rgb to skip bag reading."""
    if fused_depth is None:
        fused_depth, best_rgb = accumulate_bundle_depth(ffb_dir, K)
    if fused_depth is None:
        return 0.0, np.zeros((0, 3), dtype=np.float32)
    cam_width = int(getattr(K, 'width', 0))
    scale     = SCALE_BY_WIDTH.get(cam_width, SCALE_2D5)
    mask, z_front, z_window = _extract_mask(fused_depth, best_rgb, pipeline, K)
    depth_clipped = fused_depth.copy()
    depth_clipped[(depth_clipped > 0) & (depth_clipped > z_front + z_window)] = 0
    pts      = FFBPerceptionPipeline._project_to_3d(depth_clipped, mask, K)
    v_raw, _ = _compute_2d5_volume(pts)
    vol      = v_raw * scale if v_raw > 0 else 0.0
    print(f'    [A-vol]  {v_raw*1000:.2f}L x {scale:.2f} = {vol*1000:.2f}L')
    return vol, pts


print('Approach A functions defined.')

In [ ]:
# (approach B removed — poisson_smooth_volume was never called)

In [ ]:
# ── Approach C: Temporal accumulation + Grounding DINO → SAM2 mask ────────
# Segmentation driven by RGB semantics (Grounding DINO bbox → SAM2 mask)
# rather than depth thresholding, so it is robust to non-tarp backgrounds
# and off-centre FFB placement. Requires GDINO_AVAILABLE.
#
# Note: in-sample evaluation shows C (MAE=1.89 kg, r²=0.747) underperforms
# Approach A (MAE=1.20 kg, r²=0.863) on the current tarp-protocol dataset.
# The GDino detection path is retained as the detection fallback inside
# _patched_detect_gdino, replacing YOLO-World when depth mask fails.


def process_bundle_gdino_sam2(ffb_dir, pipeline, K,
                               fused_depth=None, best_rgb=None):
    """Approach C: fused depth + Grounding DINO → SAM2 segmentation."""
    if not GDINO_AVAILABLE:
        print('    [C]  Grounding DINO not available — skipping')
        return None, None

    if fused_depth is None:
        fused_depth, best_rgb = accumulate_bundle_depth(ffb_dir, K)
    if fused_depth is None:
        return 0.0, np.zeros((0, 3), dtype=np.float32)

    cam_width = int(getattr(K, 'width', 0))
    scale     = SCALE_BY_WIDTH.get(cam_width, SCALE_2D5)

    gdino_result = _gdino_detect(best_rgb)
    if gdino_result is None:
        print('    [C-gdino]  no detection above threshold — skipping')
        return None, None
    bbox, conf = gdino_result
    print(f'    [C-gdino]  bbox={[int(v) for v in bbox]}  conf={conf:.2f}')

    mask_rgb = pipeline._segment(best_rgb, bbox)
    dh, dw   = fused_depth.shape
    rh, rw   = best_rgb.shape[:2]
    mask = (cv2.resize(mask_rgb.astype(np.uint8), (dw, dh),
                       interpolation=cv2.INTER_NEAREST).astype(bool)
            if (rh, rw) != (dh, dw) else mask_rgb)

    if not mask.any():
        print('    [C-sam2]  empty mask — skipping')
        return None, None
    print(f'    [C-sam2]  mask={mask.mean()*100:.1f}%')

    z_front  = float(np.percentile(fused_depth[fused_depth > 0.1], 5))
    valid    = (fused_depth > 0.1) & (fused_depth < 10.0)
    margin_m = _auto_margin(fused_depth, z_front, valid)
    z_window = float(margin_m) + 0.05
    print(f'    [C-z_window]  margin={margin_m*100:.1f}cm  z_window={z_window*100:.1f}cm')

    depth_clipped = fused_depth.copy()
    depth_clipped[(depth_clipped > 0) & (depth_clipped > z_front + z_window)] = 0

    pts      = FFBPerceptionPipeline._project_to_3d(depth_clipped, mask, K)
    v_raw, _ = _compute_2d5_volume(pts)
    vol      = v_raw * scale if v_raw > 0 else 0.0
    print(f'    [C-vol]  {v_raw*1000:.2f}L x {scale:.2f} = {vol*1000:.2f}L')
    return vol, pts


print('Approach C (GDino→SAM2) functions defined.')

In [ ]:
# ── Run all approaches ─────────────────────────────────────────────────────
# Fused depth computed once per FFB, cached to fused_cache/{FFB}_fused.npz.
# Set CLEAR_CACHE = True only if max_frames changed or bags were re-recorded.

import shutil as _shutil

CLEAR_CACHE = False

if CLEAR_CACHE:
    cache_p = _pl.Path(FUSED_CACHE_DIR)
    if cache_p.exists():
        _shutil.rmtree(str(cache_p))
        print(f'Cache cleared: {cache_p}')

N_FRAMES = 15
MAX_SCAN  = 120

pipeline = make_pipeline('cuda:0' if N_GPUS >= 1 else 'cpu')
rows_v2  = []

for ffb in bundles:
    ffb_dir = os.path.join(DATA_DIR, ffb)
    print(f'\n=== {ffb} ===')
    try:
        depth_bag, _ = find_bundle_bags(ffb_dir)
        K = get_intrinsics_from_bag(depth_bag) or CameraIntrinsics.default_848x480()
        K.depth_scale = get_depth_scale(depth_bag)
    except Exception as e:
        print(f'  bag error: {e}'); continue

    g          = gt(ffb)
    cam_w      = int(getattr(K, 'width', 0))
    scale_used = SCALE_BY_WIDTH.get(cam_w, SCALE_2D5)
    row = {'ffb': ffb,
           'actual_vol_L':   g.get('Actual_Volume_L',  float('nan')),
           'actual_mass_kg': g.get('Actual_Mass_kg',   float('nan')),
           'cam_scale':      scale_used}

    # Fused depth shared between A and C
    try:
        fused_depth, best_rgb = accumulate_bundle_depth(ffb_dir, K)
    except Exception as e:
        print(f'  accum error: {e}'); fused_depth = best_rgb = None

    # v1 baseline
    try:
        br = process_bundle_multi_frame(ffb_dir, pipeline, K,
                                        n_frames=N_FRAMES, max_scan=MAX_SCAN)
        row['v1_L']  = round(br.median_volume * 1000, 3)
        row['v1_cv'] = round(100 * br.std_volume / (br.median_volume + 1e-9), 1)
    except Exception as e:
        print(f'  v1 error: {e}'); row['v1_L'] = float('nan'); row['v1_cv'] = float('nan')

    # Approach A
    try:
        vol_a, pts_a = process_bundle_accumulated(
            ffb_dir, pipeline, K, fused_depth=fused_depth, best_rgb=best_rgb)
        row['A_L']      = round(vol_a * 1000, 3)
        row['A_raw_m3'] = vol_a / scale_used if scale_used > 0 else float('nan')
    except Exception as e:
        print(f'  A error: {e}')
        row['A_L'] = float('nan'); row['A_raw_m3'] = float('nan')

    # Approach C
    try:
        result_c = process_bundle_gdino_sam2(
            ffb_dir, pipeline, K, fused_depth=fused_depth, best_rgb=best_rgb)
        if result_c[0] is not None:
            vol_c = result_c[0]
            row['C_L']      = round(vol_c * 1000, 3)
            row['C_raw_m3'] = vol_c / scale_used if scale_used > 0 else float('nan')
        else:
            row['C_L'] = float('nan'); row['C_raw_m3'] = float('nan')
    except Exception as e:
        print(f'  C error: {e}')
        row['C_L'] = float('nan'); row['C_raw_m3'] = float('nan')

    rows_v2.append(row)
    print(f'  v1={row.get("v1_L","?")}L (CV={row.get("v1_cv","?")}%)  '
          f'A={row.get("A_L","?")}L  '
          f'C={row.get("C_L","?")}L  actual={row["actual_vol_L"]}L')

print(f'\nDone: {len(rows_v2)}/{len(bundles)} bundles.')
print('Tip: set CLEAR_CACHE = False on the next run to skip bag reading.')

In [ ]:
# ── Scale recalibration on adaptive-z volumes ──────────────────────────────
# The adaptive z_window (margin+5cm) produces smaller raw volumes than the
# 25cm gate that SCALE_BY_WIDTH was originally fitted on. Refit per camera
# width so each camera's hemisphere correction is updated independently.
#
# A_raw_m3 = v_raw / scale_used  (i.e., the unscaled 2.5D integral)
# mass = scale_new * density * v_raw
# → scale_new = Σ(mass · raw) / (density · Σ(raw²))   [least-squares]

_df_rc = pd.DataFrame(rows_v2).set_index('ffb')
_valid_a = _df_rc.dropna(subset=['A_raw_m3', 'actual_mass_kg'])

SCALE_BY_WIDTH_CAL = dict(SCALE_BY_WIDTH)   # updated values go here

print('Scale recalibration (Approach A, per camera width):')
for cam_w, s_old in SCALE_BY_WIDTH.items():
    _grp = _valid_a[_valid_a['cam_scale'] == s_old]
    if len(_grp) < 2:
        print(f'  cam {cam_w}px:  n={len(_grp)} — too few samples, keeping {s_old:.4f}')
        continue
    _r = _grp['A_raw_m3'].values.astype(float)
    _m = _grp['actual_mass_kg'].values.astype(float)
    s_new = float(np.dot(_m, _r) / (DENSITY_CONSTANT * np.dot(_r, _r)))
    SCALE_BY_WIDTH_CAL[cam_w] = s_new
    print(f'  cam {cam_w}px:  n={len(_grp)}  original={s_old:.4f}  →  {s_new:.4f}  (Δ={s_new-s_old:+.4f})')

# Refit C independently per camera width
_valid_c = _df_rc.dropna(subset=['C_raw_m3', 'actual_mass_kg'])
SCALE_C_CAL = dict(SCALE_BY_WIDTH_CAL)

print('\nScale recalibration (Approach C — GDino→SAM2, per camera width):')
for cam_w, s_old in SCALE_BY_WIDTH.items():
    _grp = _valid_c[_valid_c['cam_scale'] == s_old]
    if len(_grp) < 2:
        print(f'  cam {cam_w}px:  n={len(_grp)} — using Approach A scale ({SCALE_BY_WIDTH_CAL[cam_w]:.4f})')
        continue
    _r = _grp['C_raw_m3'].values.astype(float)
    _m = _grp['actual_mass_kg'].values.astype(float)
    s_new = float(np.dot(_m, _r) / (DENSITY_CONSTANT * np.dot(_r, _r)))
    SCALE_C_CAL[cam_w] = s_new
    print(f'  cam {cam_w}px:  n={len(_grp)}  {s_old:.4f}  →  {s_new:.4f}  (Δ={s_new-s_old:+.4f})')

# Patch rows_v2 in-place using per-camera recalibrated scales
for row in rows_v2:
    cam_s = row.get('cam_scale', SCALE_2D5)
    # find which cam_w this scale belongs to
    cam_w = next((w for w, s in SCALE_BY_WIDTH.items() if s == cam_s), None)

    raw_a = row.get('A_raw_m3', float('nan'))
    if not (raw_a != raw_a) and cam_w is not None:
        row['A_L'] = round(raw_a * SCALE_BY_WIDTH_CAL[cam_w] * 1000, 3)

    raw_c = row.get('C_raw_m3', float('nan'))
    if not (raw_c != raw_c) and cam_w is not None:
        row['C_L'] = round(raw_c * SCALE_C_CAL[cam_w] * 1000, 3)

print('\nrows_v2 updated with per-camera recalibrated scales.')

In [ ]:
# ── Comparison table + mass MAE ────────────────────────────────────────────
from scipy.stats import pearsonr

df_v2 = pd.DataFrame(rows_v2).set_index('ffb')

for col in ['v1_L', 'A_L', 'C_L']:
    if col in df_v2:
        df_v2[col.replace('_L', '_err')] = (df_v2[col] - df_v2['actual_vol_L']).round(2)

display_cols = ['actual_vol_L'] + [c for c in ['v1_L','A_L','C_L',
                                                'v1_err','A_err','C_err']
                                   if c in df_v2.columns]
print(df_v2[display_cols].to_string())
df_v2.to_csv('ffb_results_v2.csv')

print('\n=== Mass metrics (excl. FFB18) ===')
ev = df_v2.dropna(subset=['actual_mass_kg']).copy()
ev = ev[ev.index != 'FFB18']

approaches = [
    ('v1_L', 'v1  15-frame IQR          '),
    ('A_L',  'A   temporal + depth mask  '),
    ('C_L',  'C   temporal + GDino→SAM2  '),
]

print(f'\n  {"Approach":<34}  {"MAE":>6}  {"MAPE":>7}  {"r":>6}  {"r²":>6}')
print(f'  {"─"*66}')

results = {}
for col, label in approaches:
    if col not in ev.columns: continue
    pred  = ev[col] / 1000.0 * DENSITY_CONSTANT
    valid = pred.notna() & ev['actual_mass_kg'].notna()
    if not valid.any(): continue
    mae  = (pred[valid] - ev.loc[valid, 'actual_mass_kg']).abs().mean()
    mape = ((pred[valid] - ev.loc[valid, 'actual_mass_kg']).abs()
            / ev.loc[valid, 'actual_mass_kg']).mean() * 100
    r, _ = pearsonr(ev.loc[valid, 'actual_mass_kg'], pred[valid]) \
           if valid.sum() > 1 else (float('nan'), None)
    r2   = r ** 2
    results[col] = dict(mae=mae, mape=mape, r=r, r2=r2)
    print(f'  {label}  {mae:>5.2f}kg  {mape:>6.1f}%  {r:>6.3f}  {r2:>6.3f}')

best_col = min(results, key=lambda k: results[k]['mae']) if results else None
labels   = {'v1_L': 'v1', 'A_L': 'A (depth mask)', 'C_L': 'C (GDino→SAM2)'}
if best_col:
    print(f'\n  --> Best by MAE: {labels.get(best_col, best_col)}  '
          f'({results[best_col]["mae"]:.2f} kg / {results[best_col]["mape"]:.1f}% / r²={results[best_col]["r2"]:.3f})')

# Volume scatter
n_cols  = sum(1 for col, _ in approaches if col in df_v2.columns)
fig, axes = plt.subplots(1, n_cols, figsize=(6 * n_cols, 5))
if n_cols == 1: axes = [axes]
df_plot = df_v2.dropna(subset=['actual_vol_L'])
x = df_plot['actual_vol_L'].values

plot_specs = [(col, lbl) for col, lbl in [
    ('v1_L', 'v1 baseline'),
    ('A_L',  'A: depth mask'),
    ('C_L',  'C: GDino→SAM2'),
] if col in df_plot.columns]

for ax, (col, title) in zip(axes, plot_specs):
    y     = df_plot[col].values
    valid = ~np.isnan(y)
    if not valid.any():
        ax.set_title(f'{title}\n(no data)'); ax.axis('off'); continue
    ax.scatter(x[valid], y[valid], s=60)
    for ffb_name, row in df_plot.iterrows():
        val = row.get(col, float('nan'))
        if not np.isnan(val):
            ax.annotate(ffb_name, (row['actual_vol_L'], val), fontsize=6,
                        xytext=(3, 3), textcoords='offset points')
    lim = [min(x[valid].min(), y[valid].min()) * 0.9,
           max(x[valid].max(), y[valid].max()) * 1.1]
    ax.plot(lim, lim, 'r--', alpha=0.4)
    if valid.sum() > 1:
        r_v, _ = pearsonr(x[valid], y[valid])
        ax.set_title(f'{title}\nr²={r_v**2:.3f}')
    else:
        ax.set_title(title)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel('Actual vol (L)'); ax.set_ylabel('Predicted vol (L)')

plt.tight_layout()
plt.savefig('comparison_v2.png', dpi=150)
plt.show()

In [ ]:
# ── Exhaustive 4/7 cross-validation ───────────────────────────────────────
# With 11 FFBs there are C(11,4)=330 possible 4-train/7-test splits.
# Each FFB appears as test in 330×7/11=210 folds.
# Total predictions = 330×7 = 2310.
#
# Note: CV refits a single scale across each training fold's camera mix.
# In-sample recalibration uses per-camera scale (see cell above).
# CV r² may therefore differ slightly from in-sample r².

from itertools import combinations
from scipy.stats import pearsonr as _pr

df_cv = pd.DataFrame(rows_v2).set_index('ffb')

valid_ffbs = df_cv.dropna(subset=['A_raw_m3', 'actual_mass_kg']).index.tolist()
n = len(valid_ffbs)
print(f'Valid FFBs for CV: {valid_ffbs}  (n={n})')

all_preds = []

for train_idx in combinations(range(n), 4):
    test_idx   = [i for i in range(n) if i not in train_idx]
    train_ffbs = [valid_ffbs[i] for i in train_idx]
    test_ffbs  = [valid_ffbs[i] for i in test_idx]

    # Refit density on 4 train FFBs
    train_nums  = [int(f.replace('FFB', '')) for f in train_ffbs]
    density_cv  = float(GT.loc[GT.index.isin(train_nums), 'True_Density_kg_L'].mean()) * 1000

    # Refit scale (least-squares) on 4 train FFBs
    r_arr = df_cv.loc[train_ffbs, 'A_raw_m3'].values.astype(float)
    m_arr = df_cv.loc[train_ffbs, 'actual_mass_kg'].values.astype(float)
    scale_cv = float(np.dot(m_arr, r_arr) / (density_cv * np.dot(r_arr, r_arr)))

    for ffb in test_ffbs:
        raw  = float(df_cv.loc[ffb, 'A_raw_m3'])
        mass = float(df_cv.loc[ffb, 'actual_mass_kg'])
        pred = raw * scale_cv * density_cv
        all_preds.append({'ffb': ffb, 'actual': mass, 'pred': pred,
                          'abs_err': abs(pred - mass),
                          'pct_err': 100 * abs(pred - mass) / mass})

df_all = pd.DataFrame(all_preds)

per_ffb = df_all.groupby('ffb').agg(
    actual=('actual', 'first'),
    pred_mean=('pred', 'mean'),
    abs_err_mean=('abs_err', 'mean'),
    pct_err_mean=('pct_err', 'mean'),
).round(3)
n_folds_each = len(list(combinations(range(n-1), 4)))
print(f'\nPer-FFB mean test-set prediction (across {n_folds_each} folds each):')
print(per_ffb.to_string())

# ── Summary metrics ────────────────────────────────────────────────────────
def _metrics(actual, pred):
    ae   = (pred - actual).abs()
    mae  = ae.mean()
    mape = (ae / actual).mean() * 100
    r, _ = _pr(actual, pred) if len(actual) > 1 else (float('nan'), None)
    return mae, mape, r ** 2

ev_all  = df_all[df_all['ffb'] != 'FFB18']
cv_mae, cv_mape, cv_r2 = _metrics(ev_all['actual'], ev_all['pred'])

ev_ffb  = per_ffb[per_ffb.index != 'FFB18']
_, _, cv_r2_ffb = _metrics(ev_ffb['actual'], ev_ffb['pred_mean'])

is_mae, is_mape, is_r2 = 1.20, 9.2, 0.929**2

print(f'\n{"":36s}  {"MAE":>6}  {"MAPE":>7}  {"r²":>6}')
print(f'{"─"*62}')
print(f'  {"In-sample A (n=11)":<34}  {is_mae:>5.2f}kg  {is_mape:>6.1f}%  {is_r2:>6.3f}')
print(f'  {"4/7 CV all predictions (n=2310)":<34}  {cv_mae:>5.2f}kg  {cv_mape:>6.1f}%  {cv_r2:>6.3f}')
print(f'  {"4/7 CV per-FFB mean (n=11)":<34}  {"—":>6}    {"—":>6}    {cv_r2_ffb:>6.3f}')
print(f'{"─"*62}')
print(f'  {"Aqil thesis (n=50, manual seg)":<34}  {"—":>6}    {"—":>6}    {"0.900":>6}')

## Results

### Per-FFB volume predictions (litres)

| FFB | Actual | v1 | A | C | v1 err | A err | C err |
|---|---|---|---|---|---|---|---|
| FFB10 | 18.0 | 18.98 | 18.70 | 17.70 | +0.98 | +0.70 | −0.30 |
| FFB11 | 14.0 | 17.40 | 15.01 | 16.34 | +3.40 | +1.01 | +2.34 |
| FFB12 | 22.0 | 22.02 | 22.56 | 23.11 | +0.02 | +0.56 | +1.11 |
| FFB17 | 14.0 | 15.76 | 11.53 | 11.02 | +1.76 | −2.47 | −2.98 |
| FFB18† | 10.0 | 20.05 | 13.19 | 12.99 | +10.05 | +3.19 | +2.99 |
| FFB19 | 20.0 | 17.30 | 18.61 | 18.07 | −2.70 | −1.39 | −1.93 |
| FFB31 | 14.0 | 15.97 | 13.41 | 12.29 | +1.97 | −0.59 | −1.71 |
| FFB32 | 10.0 | 12.24 | 8.06 | 3.86 | +2.24 | −1.95 | −6.14 |
| FFB33 | 13.0 | 14.58 | 14.92 | 14.58 | +1.58 | +1.92 | +1.57 |
| FFB34 | 14.0 | 13.83 | 12.33 | 13.75 | −0.18 | −1.67 | −0.25 |
| FFB35 | 14.0 | 10.42 | 14.05 | 14.53 | −3.58 | +0.05 | +0.53 |

† FFB18 excluded from metrics (person in frame during recording).

### In-sample metrics (excl. FFB18, n=10)

| Approach | MAE | MAPE | r | r² |
|---|---|---|---|---|
| v1 — 15-frame IQR | 1.80 kg | 13.1% | 0.828 | 0.686 |
| **A — temporal + depth mask** | **1.20 kg** | **9.2%** | **0.929** | **0.863** |
| C — temporal + GDino→SAM2 | 1.89 kg | 15.2% | 0.864 | 0.747 |

### 4/7 exhaustive cross-validation (Approach A, excl. FFB18)

C(11,4) = 330 splits · 4 train / 7 test · each FFB tested in 210 folds · 2310 total predictions.

| | MAE | MAPE | r² |
|---|---|---|---|
| All 2310 predictions | 1.67 kg | 12.4% | 0.866 |
| Per-FFB mean | — | — | **0.893** |
| Aqil thesis (n=50, manual) | — | — | 0.900 |

### Per-FFB CV breakdown (Approach A)

| FFB | Actual mass | CV pred mean | Mean abs err | MAPE |
|---|---|---|---|---|
| FFB10 | 17.0 kg | 18.91 kg | 1.91 kg | 11.2% |
| FFB11 | 14.0 kg | 15.08 kg | 1.10 kg | 7.9% |
| FFB12 | 21.6 kg | 22.70 kg | 1.33 kg | 6.2% |
| FFB17 | 13.4 kg | 11.36 kg | 2.04 kg | 15.2% |
| FFB18 | 9.6 kg | 13.46 kg | 3.86 kg | 40.2% |
| FFB19 | 19.6 kg | 18.40 kg | 1.37 kg | 7.0% |
| FFB31 | 13.8 kg | 11.58 kg | 2.22 kg | 16.1% |
| FFB32 | 9.8 kg | 6.97 kg | 2.83 kg | 28.9% |
| FFB33 | 12.0 kg | 13.14 kg | 1.15 kg | 9.6% |
| FFB34 | 12.6 kg | 10.67 kg | 1.93 kg | 15.3% |
| FFB35 | 13.0 kg | 12.23 kg | 0.86 kg | 6.6% |

### What each adaptation contributed (Approach A vs v1)

| Adaptation | Δ MAE |
|---|---|
| Temporal nanmedian depth (120 frames) | 1.80 → 1.58 kg |
| Adaptive HSV threshold + z_window | 1.58 → 1.42 kg |
| Per-camera scale recalibration | 1.42 → 1.20 kg |

### Why C underperforms A

Approach C's semantic mask (Grounding DINO → SAM2) undersegments FFB32 severely
(predicted 3.86 L vs 10.0 L actual), pulling MAE from 1.41 kg to 1.89 kg.
On the controlled tarp-protocol dataset the depth signal is a more precise
boundary cue than RGB texture. C's advantage — background-agnostic detection —
only matters when the tarp assumption breaks down.

The Grounding DINO detector is retained as the primary detection fallback inside
`_patched_detect_gdino`, replacing YOLO-World when the depth mask returns no
contour.